# Punto 1.2 — Widget interactivo con tres paneles enlazados

Se mantiene la identificación del punto 1.1, $|0\rangle \leftrightarrow \cos t$ y $|1\rangle \leftrightarrow \cos 2t$, con la multiplicación por escalares complejos
$(Ae^{ix})\cdot\cos(nt) = A\cos(nt-x)$ para $A\ge 0$, $x\in\mathbb R$. El producto hermítico construido en 1.1,

$$\langle f\mid g\rangle=\frac1\pi\Big[(f,g)_{L^2}+i\,(i\cdot f,g)_{L^2}\Big],$$

hace ortonormal a $\{|0\rangle,|1\rangle\}$, y para una onda $\psi\in V$ se reduce a

$$\langle\psi\mid\psi\rangle = \frac1\pi\int_0^{2\pi}\psi(t)^2\,dt = |\alpha|^2+|\beta|^2 .$$

El widget tiene tres paneles que muestran el mismo estado normalizado $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$:

* **(a) Plano complejo**: se arrastran $\alpha$ y $\beta$. Al mover uno, el otro ajusta su módulo para mantener $|\alpha|^2+|\beta|^2=1$ (su fase no cambia).
* **(b) Ondas** $\alpha|0\rangle$, $\beta|1\rangle$ y $\psi$ en $t\in[0,2\pi]$. También se pueden arrastrar las crestas.
* **(c) Esfera de Bloch**: el punto que corresponde a $|\psi\rangle$ (arrastrar en este panel gira la vista).

La misma lógica está implementada en JavaScript en la página web del repositorio (`index.html`), para usarla desde el navegador sin instalar Python. Este notebook es la versión de referencia y la que se reutiliza en los puntos 1.3 a 1.5.

**Requisitos:** `numpy`, `matplotlib`, `ipympl`, `ipywidgets` (ver la celda siguiente).

In [ ]:
# Si falta algo, descomente y ejecute una vez:
# %pip install numpy matplotlib ipympl ipywidgets

%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

TAU = 2 * np.pi

## Fórmulas para pasar de un panel a otro

En todas, $\alpha=|\alpha|e^{i\arg\alpha}$ y $\beta=|\beta|e^{i\arg\beta}$.

**(a) → (b)**, de los coeficientes a las ondas. Por la regla de multiplicación,
$$\alpha|0\rangle = |\alpha|\cos(t-\arg\alpha),\qquad \beta|1\rangle = |\beta|\cos(2t-\arg\beta),$$
$$\psi(t) = |\alpha|\cos(t-\arg\alpha)+|\beta|\cos(2t-\arg\beta) = \operatorname{Re}\big(\bar\alpha e^{it}+\bar\beta e^{2it}\big).$$
La cresta de $\alpha|0\rangle$ está en $t_\alpha=\arg\alpha$ con altura $|\alpha|$; las de $\beta|1\rangle$ en $t_\beta=\tfrac12\arg\beta$ y $t_\beta+\pi$ con altura $|\beta|$.

**(b) → (a)**, de la onda a los coeficientes. Los coeficientes son los productos hermíticos de 1.1 con los elementos de la base:
$$\alpha = \langle 0\mid\psi\rangle = \frac1\pi\int_0^{2\pi}\psi(t)\,e^{it}\,dt,\qquad \beta = \langle 1\mid\psi\rangle = \frac1\pi\int_0^{2\pi}\psi(t)\,e^{2it}\,dt .$$
En efecto, como $i\cdot\cos t=\sin t$, la definición del producto hermítico da $\langle 0\mid\psi\rangle=\frac1\pi\big[(\cos t,\psi)_{L^2}+i(\sin t,\psi)_{L^2}\big]=\frac1\pi\int_0^{2\pi}\psi(t)(\cos t+i\sin t)\,dt=\frac1\pi\int_0^{2\pi}\psi(t)e^{it}\,dt$, y lo mismo ocurre con $\langle 1\mid\psi\rangle$ usando $\cos 2t$ y $\sin 2t$. Leídos en las crestas: $|\alpha|$ y $|\beta|$ son las alturas, $\arg\alpha=t_\alpha$ y $\arg\beta=2t_\beta$.

**(a) → (c)**, de los coeficientes a la esfera. Salvo fase global, $|\psi\rangle \sim \cos\frac\theta2|0\rangle + e^{i\varphi}\sin\frac\theta2|1\rangle$ con
$$\cos\tfrac\theta2=|\alpha|,\quad \sin\tfrac\theta2=|\beta|\quad(\theta\in[0,\pi]),\qquad \varphi = \arg\beta-\arg\alpha,$$
$$\vec r = (\sin\theta\cos\varphi,\ \sin\theta\sin\varphi,\ \cos\theta) = \big(2\operatorname{Re}(\bar\alpha\beta),\ 2\operatorname{Im}(\bar\alpha\beta),\ |\alpha|^2-|\beta|^2\big).$$
Si $\alpha=0$ o $\beta=0$, el estado está en un polo y $\varphi$ no interviene. (En el código, $\theta$ se calcula con `arctan2(|β|, |α|)`, que no falla cuando $\alpha=0$.)

**(c) → (a)**, de la esfera a los coeficientes:
$$\alpha = e^{i\gamma}\cos\tfrac\theta2,\qquad \beta = e^{i(\gamma+\varphi)}\sin\tfrac\theta2,$$
donde $\gamma$ es una fase global arbitraria: la esfera no la registra.

**(b) → (c)**, de la onda a la esfera:
$$\cos\theta=|\alpha|^2-|\beta|^2=(\text{altura de }\alpha|0\rangle)^2-(\text{altura de }\beta|1\rangle)^2,\qquad \varphi\equiv 2t_\beta-t_\alpha\pmod{2\pi}.$$
La latitud mide cómo se reparte la norma entre las frecuencias 1 y 2; la longitud, el desfase entre crestas contado en la escala de la frecuencia 2 (no es la simple diferencia $t_\beta-t_\alpha$, porque las frecuencias son distintas).

**(c) → (b)**, de la esfera a la onda:
$$\psi(t)=\cos\tfrac\theta2\,\cos(t-\gamma)+\sin\tfrac\theta2\,\cos(2t-\gamma-\varphi).$$

**Fase global y desplazamientos en el tiempo.** Multiplicar el estado por $e^{i\gamma}$ corre la onda $\alpha|0\rangle$ una distancia $\gamma$ y la onda $\beta|1\rangle$ solo $\gamma/2$, de modo que $\psi$ cambia de forma, pero el punto de Bloch no se mueve. En cambio, desplazar $\psi$ en el tiempo una cantidad $\tau$ multiplica $\alpha$ por $e^{i\tau}$ y $\beta$ por $e^{2i\tau}$: esto suma $\tau$ a $\varphi$ y gira el punto de Bloch alrededor del eje $z$.

La celda siguiente implementa estas fórmulas como funciones (reutilizables en el punto 1.5(c)) y verifica numéricamente que todos los pasos son consistentes.

In [ ]:
def envolver(p):
    # Lleva un ángulo a (-π, π].
    return float(np.angle(np.exp(1j * p)))


def ondas(alpha, beta, t):
    # (a)→(b): devuelve α|0⟩, β|1⟩ y ψ evaluadas en t.
    f = abs(alpha) * np.cos(t - np.angle(alpha))
    g = abs(beta) * np.cos(2 * t - np.angle(beta))
    return f, g, f + g


def coeficientes(psi, n_puntos=2000):
    # (b)→(a): α = (1/π)∫ψ e^{it} dt, β = (1/π)∫ψ e^{2it} dt, con suma de Riemann.
    t = np.linspace(0, TAU, n_puntos, endpoint=False)
    v, dt = psi(t), TAU / n_puntos
    alpha = np.sum(v * np.exp(1j * t)) * dt / np.pi
    beta = np.sum(v * np.exp(2j * t)) * dt / np.pi
    return complex(alpha), complex(beta)


def producto_L2(f, g, n_puntos=2000):
    # (f, g)_{L²} = ∫ f g dt, con suma de Riemann.
    t = np.linspace(0, TAU, n_puntos, endpoint=False)
    return float(np.sum(f(t) * g(t)) * (TAU / n_puntos))


def norma2(psi, n_puntos=2000):
    # ⟨ψ|ψ⟩ = (1/π)∫ψ² dt.
    return producto_L2(psi, psi, n_puntos) / np.pi


def bloch(alpha, beta):
    # (a)→(c): devuelve θ, φ y el vector r = (2Re ᾱβ, 2Im ᾱβ, |α|²−|β|²).
    n = np.sqrt(abs(alpha) ** 2 + abs(beta) ** 2)
    alpha, beta = alpha / n, beta / n
    theta = 2 * np.arctan2(abs(beta), abs(alpha))
    phi = envolver(np.angle(beta) - np.angle(alpha))
    ab = np.conj(alpha) * beta
    r = np.array([2 * ab.real, 2 * ab.imag, abs(alpha) ** 2 - abs(beta) ** 2])
    return float(theta), phi, r


def desde_bloch(theta, phi, gamma=0.0):
    # (c)→(a): α = e^{iγ}cos(θ/2), β = e^{i(γ+φ)}sin(θ/2).
    return np.exp(1j * gamma) * np.cos(theta / 2), np.exp(1j * (gamma + phi)) * np.sin(theta / 2)


def onda_desde_bloch(theta, phi, t, gamma=0.0):
    # (c)→(b): ψ(t) = cos(θ/2) cos(t − γ) + sin(θ/2) cos(2t − γ − φ).
    return np.cos(theta / 2) * np.cos(t - gamma) + np.sin(theta / 2) * np.cos(2 * t - gamma - phi)


def crestas(alpha, beta):
    # Posición de la cresta de α|0⟩ y de la primera cresta de β|1⟩ en [0, 2π).
    return np.angle(alpha) % TAU, (np.angle(beta) / 2) % np.pi


# Verificación con estados aleatorios
rng = np.random.default_rng(0)
t = np.linspace(0, TAU, 500)
for _ in range(200):
    z = rng.normal(size=2) + 1j * rng.normal(size=2)
    a, b = z / np.linalg.norm(z)
    psi = lambda s: ondas(a, b, s)[2]

    # (b)→(a) invierte (a)→(b), y coincide con el producto hermítico de 1.1: α = ⟨0|ψ⟩, β = ⟨1|ψ⟩
    assert np.allclose(coeficientes(psi), (a, b))
    for n, coef in [(1, a), (2, b)]:
        c, s_ = (lambda s: np.cos(n * s)), (lambda s: np.sin(n * s))
        assert np.isclose((producto_L2(c, psi) + 1j * producto_L2(s_, psi)) / np.pi, coef)
    assert np.isclose(norma2(psi), 1)                                    # ‖ψ‖² = |α|²+|β|²

    # (a)→(c) y (c)→(a)
    th, ph, r = bloch(a, b)
    assert np.allclose(r, [np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)])
    assert np.isclose(np.cos(th / 2), abs(a))
    assert np.allclose(desde_bloch(th, ph, gamma=np.angle(a)), (a, b))

    # (c)→(b): la onda reconstruida desde la esfera coincide con ψ (eligiendo γ = arg α)
    assert np.allclose(onda_desde_bloch(th, ph, t, gamma=np.angle(a)), psi(t))

    # (b)→(c): φ ≡ 2 t_β − t_α (mód 2π) y cos θ = |α|² − |β|²
    ta, tb = crestas(a, b)
    assert np.isclose(envolver(2 * tb - ta), ph)
    assert np.isclose(np.cos(th), abs(a) ** 2 - abs(b) ** 2)

    # Fase global: el punto de Bloch no cambia, pero la onda ψ sí cambia de forma
    g = rng.uniform(-np.pi, np.pi)
    assert np.allclose(bloch(np.exp(1j * g) * a, np.exp(1j * g) * b)[2], r)

    # Desplazamiento en el tiempo τ: α → α e^{iτ}, β → β e^{2iτ}, giro de Bloch en τ alrededor de z
    tau = rng.uniform(0, TAU)
    a2, b2 = coeficientes(lambda s: psi(s - tau))
    assert np.allclose((a2, b2), (a * np.exp(1j * tau), b * np.exp(2j * tau)))
    giro = np.array([[np.cos(tau), -np.sin(tau), 0], [np.sin(tau), np.cos(tau), 0], [0, 0, 1]])
    assert np.allclose(bloch(a2, b2)[2], giro @ r)

print('Todas las fórmulas de paso son consistentes ✓')

## El widget

Se ejecuta la celda y se interactúa con la figura:

* **Panel (a)**: arrastre el punto azul ($\alpha$) o el naranja ($\beta$).
* **Panel (b)**: arrastre la cresta de $\alpha|0\rangle$ o de $\beta|1\rangle$: la posición en $t$ fija la fase y la altura el módulo.
* **Panel (c)**: arrastre para girar la esfera.
* **Fase global $\gamma=\arg\alpha$**: multiplica $\alpha$ y $\beta$ por $e^{i\gamma}$. Ambos puntos giran el mismo ángulo en (a); en (b) la onda $\alpha|0\rangle$ se corre $\gamma$ y la onda $\beta|1\rangle$ solo $\gamma/2$, así que $\psi$ cambia de forma; el punto de Bloch no se mueve.

Debajo de la figura se muestran los valores numéricos, incluidas las integrales de (b)→(a) calculadas sobre la onda dibujada.

In [ ]:
COL_A, COL_B, COL_PSI, COL_P = '#2563eb', '#d9480f', '#1d1d1f', '#7c3aed'
S2 = 1 / np.sqrt(2)
ESTADOS = {
    '|0⟩': (1, 0), '|1⟩': (0, 1),
    '|+⟩': (S2, S2), '|−⟩': (S2, -S2),
    '|+i⟩': (S2, 1j * S2), '|−i⟩': (S2, -1j * S2),
}


class WidgetQubit:
    # Estado guardado como módulos y fases por separado, para no perder la fase cuando un módulo es 0.

    def __init__(self, alpha=S2, beta=S2 * np.exp(1j * np.pi / 3)):
        self.t = np.linspace(0, TAU, 400)
        self._arrastre = None
        self._sincronizando = False
        self.fijar_estado(alpha, beta)
        self._construir_figura()
        self._construir_controles()
        self.actualizar()

    @property
    def alpha(self):
        return self.ra * np.exp(1j * self.pa)

    @property
    def beta(self):
        return self.rb * np.exp(1j * self.pb)

    # ---------- cambios de estado (mantienen |α|² + |β|² = 1) ----------
    def fijar_estado(self, alpha, beta):
        n = np.sqrt(abs(alpha) ** 2 + abs(beta) ** 2)
        self.ra, self.pa = abs(alpha) / n, envolver(np.angle(alpha))
        self.rb, self.pb = abs(beta) / n, envolver(np.angle(beta))

    def fijar_alpha(self, r, p):
        self.ra, self.pa = float(np.clip(r, 0, 1)), envolver(p)
        self.rb = np.sqrt(1 - self.ra ** 2)

    def fijar_beta(self, r, p):
        self.rb, self.pb = float(np.clip(r, 0, 1)), envolver(p)
        self.ra = np.sqrt(1 - self.rb ** 2)

    def crestas(self):
        ta = self.pa % TAU
        tb = (self.pb / 2) % np.pi
        return {'a': [(ta, self.ra)], 'b': [(tb, self.rb), (tb + np.pi, self.rb)]}

    # ---------- figura ----------
    def _construir_figura(self):
        with plt.ioff():
            self.fig = plt.figure(figsize=(13, 4.6))
        self.ax_c = self.fig.add_subplot(1, 3, 1)
        self.ax_w = self.fig.add_subplot(1, 3, 2)
        self.ax_b = self.fig.add_subplot(1, 3, 3, projection='3d')
        s = np.linspace(0, TAU, 200)

        # (a) plano complejo
        ax = self.ax_c
        ax.plot(np.cos(s), np.sin(s), color='0.55', lw=1)
        ax.axhline(0, color='0.7', lw=0.8)
        ax.axvline(0, color='0.7', lw=0.8)
        ax.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), xlabel='Re', ylabel='Im', aspect='equal')
        ax.grid(alpha=0.3)
        ax.set_title('(a) Plano complejo: arrastre α o β')
        self.circ, self.vec, self.pt, self.lbl = {}, {}, {}, {}
        for k, col, nombre in [('a', COL_A, 'α'), ('b', COL_B, 'β')]:
            self.circ[k], = ax.plot([], [], ':', color=col, lw=1, alpha=0.6)
            self.vec[k], = ax.plot([], [], '-', color=col, lw=2)
            self.pt[k], = ax.plot([], [], 'o', color=col, ms=11, mec='w', mew=1.5, zorder=5)
            self.lbl[k] = ax.text(0, 0, nombre, color=col, fontsize=14, fontweight='bold')

        # (b) ondas
        ax = self.ax_w
        ax.axhline(0, color='0.6', lw=0.8)
        ax.set(xlim=(0, TAU), ylim=(-1.6, 1.6), xlabel='t')
        ax.set_xticks(np.arange(5) * np.pi / 2, ['0', 'π/2', 'π', '3π/2', '2π'])
        ax.grid(alpha=0.3)
        ax.set_title('(b) Ondas: arrastre las crestas')
        self.onda_a, = ax.plot(self.t, 0 * self.t, '--', color=COL_A, lw=1.8, label=r'$\alpha|0\rangle$')
        self.onda_b, = ax.plot(self.t, 0 * self.t, '--', color=COL_B, lw=1.8, label=r'$\beta|1\rangle$')
        self.onda_psi, = ax.plot(self.t, 0 * self.t, '-', color=COL_PSI, lw=2.8, label=r'$\psi$')
        self.cresta = {
            'a': ax.plot([], [], 'o', color=COL_A, ms=9, mec='w', mew=1.5, zorder=5)[0],
            'b': ax.plot([], [], 'o', color=COL_B, ms=9, mec='w', mew=1.5, zorder=5)[0],
        }
        ax.legend(loc='lower right', fontsize=9, ncol=3)

        # (c) esfera de Bloch
        ax = self.ax_b
        u, v = np.mgrid[0:TAU:36j, 0:np.pi:18j]
        ax.plot_wireframe(np.cos(u) * np.sin(v), np.sin(u) * np.sin(v), np.cos(v), color='0.88', lw=0.5)
        ax.plot(np.cos(s), np.sin(s), 0 * s, color='0.55', lw=1)
        for eje in np.eye(3):
            ax.plot(*np.array([-eje, eje]).T, color='0.6', lw=0.8)
        for pos, txt in [((0, 0, 1.25), r'$|0\rangle$'), ((0, 0, -1.35), r'$|1\rangle$'),
                         ((1.35, 0, 0), r'$|+\rangle$'), ((-1.35, 0, 0), r'$|-\rangle$'),
                         ((0, 1.35, 0), r'$|{+i}\rangle$'), ((0, -1.35, 0), r'$|{-i}\rangle$')]:
            ax.text(*pos, txt, ha='center', va='center', color='0.3')
        self.vec_bloch, = ax.plot([0, 0], [0, 0], [0, 0], color=COL_P, lw=2.5)
        self.pt_bloch, = ax.plot([0], [0], [0], 'o', color=COL_P, ms=9)
        self.proy_bloch, = ax.plot([], [], [], ':', color=COL_P, lw=1)
        ax.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), zlim=(-1.1, 1.1))
        ax.set_box_aspect((1, 1, 1), zoom=1.35)
        ax.set_axis_off()
        ax.view_init(elev=20, azim=30)
        ax.set_title('(c) Esfera de Bloch (arrastre para girar)')

        self.fig.tight_layout()
        if hasattr(self.fig.canvas, 'header_visible'):
            self.fig.canvas.header_visible = False
        self.fig.canvas.mpl_connect('button_press_event', self._presionar)
        self.fig.canvas.mpl_connect('motion_notify_event', self._mover)
        self.fig.canvas.mpl_connect('button_release_event', self._soltar)

    # ---------- controles ----------
    def _construir_controles(self):
        botones = []
        for nombre, (a, b) in ESTADOS.items():
            btn = widgets.Button(description=nombre, layout=widgets.Layout(width='64px'))
            btn.on_click(lambda _, a=a, b=b: self._ir_a(a, b))
            botones.append(btn)
        self.gamma = widgets.FloatSlider(
            value=0, min=-180, max=180, step=1, description='Fase global γ = arg α (°)',
            style={'description_width': 'initial'}, layout=widgets.Layout(width='480px'))
        self.gamma.observe(self._cambio_gamma, names='value')
        self.lectura = widgets.HTML()
        self.ui = widgets.VBox([
            widgets.HBox([widgets.Label('Estados:')] + botones),
            self.gamma, self.fig.canvas, self.lectura])

    def _ir_a(self, a, b):
        self.fijar_estado(a, b)
        self.actualizar()

    def _cambio_gamma(self, cambio):
        if self._sincronizando:
            return
        d = np.deg2rad(cambio['new'] - cambio['old'])
        self.pa, self.pb = envolver(self.pa + d), envolver(self.pb + d)
        self.actualizar(sincronizar_gamma=False)

    # ---------- ratón ----------
    def _mas_cercano(self, ax, ev, puntos):
        # Devuelve la clave del punto más cercano al clic (en píxeles), o None si está a más de 15 px.
        mejor, dmin = None, 15
        for k, lista in puntos.items():
            for xy in lista:
                px, py = ax.transData.transform(xy)
                d = np.hypot(px - ev.x, py - ev.y)
                if d < dmin:
                    mejor, dmin = k, d
        return mejor

    def _presionar(self, ev):
        if ev.inaxes is self.ax_c:
            puntos = {'a': [(self.alpha.real, self.alpha.imag)], 'b': [(self.beta.real, self.beta.imag)]}
            k = self._mas_cercano(self.ax_c, ev, puntos)
            self._arrastre = ('c', k) if k else None
        elif ev.inaxes is self.ax_w:
            k = self._mas_cercano(self.ax_w, ev, self.crestas())
            self._arrastre = ('w', k) if k else None

    def _mover(self, ev):
        if not self._arrastre or ev.xdata is None:
            return
        panel, k = self._arrastre
        if panel == 'c' and ev.inaxes is self.ax_c:
            z = complex(ev.xdata, ev.ydata)
            fase = np.angle(z) if abs(z) > 1e-9 else (self.pa if k == 'a' else self.pb)
            (self.fijar_alpha if k == 'a' else self.fijar_beta)(abs(z), fase)
        elif panel == 'w' and ev.inaxes is self.ax_w:
            t = float(np.clip(ev.xdata, 0, TAU))
            if k == 'a':
                self.fijar_alpha(ev.ydata, t)
            else:
                self.fijar_beta(ev.ydata, 2 * t)
        else:
            return
        self.actualizar()

    def _soltar(self, ev):
        self._arrastre = None

    # ---------- dibujo ----------
    def actualizar(self, sincronizar_gamma=True):
        a, b = self.alpha, self.beta
        s = np.linspace(0, TAU, 200)

        # (a)
        for k, z, r in [('a', a, self.ra), ('b', b, self.rb)]:
            self.circ[k].set_data(r * np.cos(s), r * np.sin(s))
            self.vec[k].set_data([0, z.real], [0, z.imag])
            self.pt[k].set_data([z.real], [z.imag])
            self.lbl[k].set_position((z.real + 0.06, z.imag + 0.06))

        # (b)
        f, g, psi = ondas(a, b, self.t)
        self.onda_a.set_ydata(f)
        self.onda_b.set_ydata(g)
        self.onda_psi.set_ydata(psi)
        for k, lista in self.crestas().items():
            self.cresta[k].set_data(*zip(*lista))

        # (c)
        th, ph, r = bloch(a, b)
        self.vec_bloch.set_data_3d([0, r[0]], [0, r[1]], [0, r[2]])
        self.pt_bloch.set_data_3d([r[0]], [r[1]], [r[2]])
        self.proy_bloch.set_data_3d([r[0], r[0], 0], [r[1], r[1], 0], [r[2], 0, 0])

        if sincronizar_gamma:
            self._sincronizando = True
            self.gamma.value = round(np.rad2deg(self.pa))
            self._sincronizando = False
        self._escribir_lectura(th, ph, r)
        self.fig.canvas.draw_idle()

    def _escribir_lectura(self, th, ph, r):
        a, b = self.alpha, self.beta
        ai, bi = coeficientes(lambda t: ondas(a, b, t)[2])
        n2 = norma2(lambda t: ondas(a, b, t)[2])
        c = lambda z: f'{z.real:+.3f} {"+" if z.imag >= 0 else "−"} {abs(z.imag):.3f}i'
        deg = lambda x: f'{np.rad2deg(x):.1f}°'
        celda = 'style="padding:2px 18px 2px 0; vertical-align:top"'
        self.lectura.value = f'''
        <table style="font-family:monospace; font-size:13px"><tr>
          <td {celda}><b style="color:{COL_A}">α</b> = {c(a)}<br>|α| = {self.ra:.3f}, arg α = {deg(self.pa)}<br>P(0) = |α|² = {self.ra**2:.3f}</td>
          <td {celda}><b style="color:{COL_B}">β</b> = {c(b)}<br>|β| = {self.rb:.3f}, arg β = {deg(self.pb)}<br>P(1) = |β|² = {self.rb**2:.3f}</td>
          <td {celda}><b>Onda → coeficientes</b> (Riemann, N = 2000)<br>(1/π)∫ψ e<sup>it</sup> dt = {c(ai)}<br>(1/π)∫ψ e<sup>2it</sup> dt = {c(bi)}<br>(1/π)∫ψ² dt = {n2:.3f}</td>
          <td {celda}><b style="color:{COL_P}">Bloch</b><br>θ = {deg(th)}, φ = arg β − arg α = {deg(ph)}<br>r = ({r[0]:+.3f}, {r[1]:+.3f}, {r[2]:+.3f})</td>
        </tr></table>'''


w = WidgetQubit()
display(w.ui)

## Leer el estado actual

Después de mover el widget, el estado queda disponible en `w.alpha` y `w.beta`. Sirve para comparar con los vectores de Bloch que calcule Qiskit en el punto 1.5(c).

In [ ]:
th, ph, r = bloch(w.alpha, w.beta)
print(f'α = {w.alpha:.4f},  β = {w.beta:.4f}')
print(f'θ = {np.rad2deg(th):.2f}°,  φ = {np.rad2deg(ph):.2f}°,  r = {np.round(r, 4)}')